In [1]:
import pandas as pd
from openai import OpenAI
import os

In [2]:
api_key=os.environ["OPENAI_API_KEY"]

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter #This split by new line
from langchain_text_splitters import RecursiveCharacterTextSplitter  #This split by character not necessarily by new line i.e \n

#Third example of creating and splitting documents

#url="https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

#Adidas_Global_url="https://drive.google.com/uc?export=download&id=1QEV9zVQCmyTXmsR9rvkop2amsCySBkRZ"
US_Adidas_url="https://drive.google.com/uc?export=download&id=1TBALzBg6VCadt9FFcVykjv07bGllDJsg"

df = pd.read_csv(US_Adidas_url, dtype={"availability_units": "string"})
display(df)

doc3=[Document(page_content=", ".join(f"{col}: {row[col]}" for col in df.columns) , metadata={"row": i+1}) for i, row in df.iterrows()]

text_splitter3=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks3=text_splitter3.split_documents(doc3)


,snapshot_date,country_code,product_name,model_number,currency,price_local,sale_price_local,gender_segment,size_label,category,...,rating_count,badge_texts,product_url,canonical_url,image_url,best_for_ids,seen_market_count,seen_markets,market_record_source,size_record_source
0,2026-02-28,US,3-STRIPES COACH JACKET,ZG695,USD,80.0,NaN,M,2XL,Lifestyle,...,9.0,Extra 25% with CODE: DEAL,https://www.adidas.com/us/3-stripes-coach-jack...,https://www.adidas.com/us/3-stripes-coach-jack...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
1,2026-02-28,US,3-STRIPES COACH JACKET,ZG695,USD,80.0,NaN,M,L,Lifestyle,...,9.0,Extra 25% with CODE: DEAL,https://www.adidas.com/us/3-stripes-coach-jack...,https://www.adidas.com/us/3-stripes-coach-jack...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
2,2026-02-28,US,3-STRIPES COACH JACKET,ZG695,USD,80.0,NaN,M,M,Lifestyle,...,9.0,Extra 25% with CODE: DEAL,https://www.adidas.com/us/3-stripes-coach-jack...,https://www.adidas.com/us/3-stripes-coach-jack...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
3,2026-02-28,US,3-STRIPES COACH JACKET,ZG695,USD,80.0,NaN,M,S,Lifestyle,...,9.0,Extra 25% with CODE: DEAL,https://www.adidas.com/us/3-stripes-coach-jack...,https://www.adidas.com/us/3-stripes-coach-jack...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
4,2026-02-28,US,3-STRIPES COACH JACKET,ZG695,USD,80.0,NaN,M,XL,Lifestyle,...,9.0,Extra 25% with CODE: DEAL,https://www.adidas.com/us/3-stripes-coach-jack...,https://www.adidas.com/us/3-stripes-coach-jack...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5741,2026-02-28,US,adidas Z.N.E. Tee,KTJ83,USD,27.0,27.0,W,L,Lifestyle,...,NaN,NaN,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
5742,2026-02-28,US,adidas Z.N.E. Tee,KTJ83,USD,27.0,27.0,W,M,Lifestyle,...,NaN,NaN,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
5743,2026-02-28,US,adidas Z.N.E. Tee,KTJ83,USD,27.0,27.0,W,S,Lifestyle,...,NaN,NaN,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact
5744,2026-02-28,US,adidas Z.N.E. Tee,KTJ83,USD,27.0,27.0,W,XL,Lifestyle,...,NaN,NaN,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,https://www.adidas.com/us/adidas-z.n.e.-tee/JE...,"https://assets.adidas.com/images/w_600,f_auto,...",[],1,US,api,api_exact


In [4]:
#print(df["canonical_url"])

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()  #convert text into numerical values, using openAI trained data for vector embeddings
vectorstore= FAISS.from_documents(chunks3,embeddings)
retriever= vectorstore.as_retriever()

In [7]:
#This is building a RAG pipeline using LCEL (LangChain Expression Language),
#basically a clean, composable way to define how data flows through your system.

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

"""User Question
     ↓
Pass through + Retrieve context
     ↓
Insert into prompt
     ↓
LLM generates answer
     ↓
Convert to clean string"""

#Creatiing LLMs

llm = ChatOpenAI(model="gpt-4.1", temperature=0)


prompt= PromptTemplate.from_template(
    """Answer the following question:
    {question}
    Based on the following context:
    {context}
    """
)

chain= (
    {"context": retriever, "question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)


In [ ]:
from langchain_core.runnables import RunnableWithMessageHistory
#from langchain.memory import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

#Define a second prompt which accomodates chat history


prompt2= PromptTemplate.from_template("""
You are a shop assistant.

A store catalogue: {context} is called Adidas store.

chat history:
{chat_history}

costomer question:
{question}


""")

#Using a new cjhain to accomaodate the chat history inside an input mapping
chain2 = (
    {
        "context": itemgetter("question") | retriever | (lambda docs: "\n".join(d.page_content for d in docs)),
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history"),
    }
    | prompt2
    | llm
    | StrOutputParser()
)

#define a dictionary that will contain a message history

store={}

def get_message_history(session_id:str):
    if session_id not in store:
        store[session_id]= InMemoryChatMessageHistory()
    return store[session_id]


#wrap the chain2 with RunnableWithMessageHistory, to get chai with memory

chain_with_memory= RunnableWithMessageHistory(
    chain2,
    get_message_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)


while True:
    user_input=input("user: ")
    if user_input.lower() in ["quit","exit"]:
        break

    #Invoke with memory

    response2 = chain_with_memory.invoke(
        {"question":user_input},
        config={"configurable":{"session_id":"user1"}}
    )

    print(response2)

    

user:  are there any shoes to buy?


Yes! We currently have the Runfalcon 5 Running Shoes available at our Adidas store. These shoes are best for everyday use, comfort, and walking. Here’s what they look like:

![Runfalcon 5 Running Shoes](https://assets.adidas.com/images/w_600,f_auto,q_auto/39508d2baa194dc391c2820ea03228d7_9366/Runfalcon_5_Running_Shoes_Turquoise_JR3088_HM1.jpg)

Would you like to know more details or check available sizes?


user:  yes please


Great! Here are more details about the Runfalcon 5 Running Shoes:

**Product:** Runfalcon 5 Running Shoes  
**Best for:** Everyday use, comfort, and walking  
**Color:** Turquoise  
**Features:**
- Lightweight and breathable mesh upper
- Cushioned midsole for all-day comfort
- Durable rubber outsole for good grip
- Classic Adidas 3-stripes design

**Available Sizes:**  
We typically carry a full range of men’s and women’s US sizes. Would you like to tell me your size so I can check availability for you?

**Price:**  
Would you like to know the price as well?

Let me know if you’d like to see more photos or need help with sizing!


user:  size 8


Thank you for your interest! For the Runfalcon 5 Running Shoes, I will check if size 8 is available.

Currently, I do not have the specific stock information for size 8 in the Runfalcon 5 Running Shoes. Would you like me to check with our inventory or suggest similar shoes in size 8? If you’re interested in other products, we also have the Tennis ClimaCool Ergo Shorts available in various sizes.

Let me know how you’d like to proceed!


user:  yes please


Thank you! I will check the availability of the Runfalcon 5 Running Shoes in size 8 for you.

Currently, I do not have real-time stock information, but I recommend visiting our Adidas store or checking our website for the most up-to-date availability in size 8. If you’d like, I can also suggest similar shoes in size 8 or help you with other products, such as the SKATEBOARDING PINTUCK PANTS 2.0.

Would you like to see similar shoes, or are you interested in other items from our store? Let me know how I can assist you further!


user:  please uuggest similar shoes of size 8


Thank you for your patience! Based on our current catalogue, I have found a similar option for you:

**adidas Pixar Toy Story VL Court 3.0 Shoes (Infants)**
- **Available Sizes:** 8.5K, 9K, 9.5K, 10K (These are infant/kids' sizes, not adult size 8)
- **Availability:** In stock (15 units available)
- **Product Link:** [View on adidas.com](https://www.adidas.com/us/adidas-pixar-toy-story-vl-court-3.0-shoes-infants/JQ4381.html)

Unfortunately, at the moment, I do not have an exact match for adult size 8 in our current catalogue. The available shoes are in kids' sizes.

Would you like to see more options, or are you interested in other types of footwear or products? Let me know how I can assist you further!


In [23]:
from typing import TypedDict, List

#define state
class State(TypedDict):
    message: List[str]

#define nodes

def chatbot_node(state: State):
    user_message=state["message"][-1]

    response = f"Echo: {user_message}"

    return {"message": state["message"]+ [response]}

#build the graph

from langgraph.graph import StateGraph

graph= StateGraph(State)

graph.add_node("chatbot",chatbot_node)
graph.set_entry_point("chatbot")
graph.set_finish_point("chatbot")

app =graph.compile()

app.invoke({"message": ["how are you?"]})

{'message': ['how are you?', 'Echo: how are you?']}